In [2]:

# Install the packages we'll need today
# (uncomment and run once)
# !pip install requests python-dotenv openai

In [6]:
import os
from dotenv import load_dotenv
import requests

load_dotenv()

True

# The Free Alternative: Groq + Llama 3
OpenAI works great, but it costs money. If you're learning, experimenting, or building a side project — there's a beautiful free option:

## Meet Groq
Groq (not Elon's Grok — different company!) runs open-source models (Llama 3, Mixtral, Gemma, DeepSeek) on their custom hardware. They give a generous free tier — thousands of free requests per day, no credit card required.

And here's the magic: they copied OpenAI's API format exactly. So the same code we just wrote works on Groq with a 2-line change.

Getting your Groq API key
1. Go to https://console.groq.com.
2. Sign in with Google or GitHub (no credit card needed).
3. In the left sidebar, click API Keys.
4. Click + Create API Key, name it, copy the key (starts with gsk_...).
5. Add it to your .env:

GROQ_API_KEY=gsk_your-actual-key-here

In [5]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "❌ No Groq key found. Add GROQ_API_KEY to your .env"
print(f" Groq key loaded. Starts with: {GROQ_API_KEY[:8]}...")

 Groq key loaded. Starts with: gsk_bPiP...


1. <b> The URL </b> → <a>https://api.groq.com/openai/v1/chat/completions</a><br>
2. <b> The model </b> → llama-3.3-70b-versatile (a free open-source model)

In [7]:
url = "https://api.groq.com/openai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {GROQ_API_KEY}",
    "Content-Type": "application/json",
}

body = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "system", "content": "You are a helpful Python tutor. Keep replies under 3 sentences."},
        {"role": "user", "content": "What is an API key with 3 points ?"},
    ],
}

response = requests.post(url, headers=headers, json=body, timeout=30)
data = response.json()

print(" Reply from Llama:", data["choices"][0]["message"]["content"])
print(f"\n Tokens used: {data['usage']['total_tokens']}  (cost: $0.00 — free tier)")

 Reply from Llama: An API key is a unique code used to authenticate and authorize access to a web service or application. Here are three key points about API keys: 
1. **Security**: API keys are used to secure data and prevent unauthorized access.
2. **Access control**: They control what actions can be performed by a user or application.
3. **Tracking usage**: API keys help track usage and monitor activity for billing or analytics purposes.

 Tokens used: 146  (cost: $0.00 — free tier)


<h2>Bonus — Groq with the openai SDK (one-line change!)</h2>
<h5>Because Groq is OpenAI-compatible, you can use the OpenAI Python package itself to talk to Groq. You just point it at a different base_url. Same client, same methods, free models.</h5>

In [8]:
from openai import OpenAI

In [12]:
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)



response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a helpful Agentic AI tutor. Keep replies under 3-5 sentences."},
        {"role": "user", "content": "What is Agentic Ai give in 3-5 points ?"},
    ],
)

print("Reply:", response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

Reply: Agentic AI refers to artificial intelligence that can act independently and make decisions. Key points about Agentic AI include: 
* It has autonomy to perform tasks without human intervention.
* It can learn from experiences and adapt to new situations.
* Agentic AI can interact with its environment and make decisions based on its goals and objectives.

Tokens used: 133


# 🧠 Compare Free Open-Source Models on Groq

Groq lets you switch between multiple open-source LLMs by changing a single model name.

Think of it as swapping the **brain** while keeping the same application code.

---

## 🚀 Available Models

| Model | Creator | Strength | Speed |
|---------|----------|----------|----------|
| 🦙 `llama-3.3-70b-versatile` | Meta | Best overall performance | ⚡⚡⚡ |
| 🦙 `llama-3.1-8b-instant` | Meta | Ultra-fast responses | ⚡⚡⚡⚡⚡ |
| 🌪️ `mixtral-8x7b-32768` | Mistral AI | Long-context tasks (32k tokens) | ⚡⚡⚡ |
| 💎 `gemma2-9b-it` | Google | Concise, instruction-following | ⚡⚡⚡⚡ |
| 🧩 `deepseek-r1-distill-llama-70b` | DeepSeek | Reasoning & mathematics | ⚡⚡ |

⚠️ Groq updates this list often. Check https://console.groq.com/docs/models for the current models if any name fails.

In [13]:
models_to_try = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "gemma2-9b-it",
]

question = "In 1 sentence, what makes Python a great language for beginners?"

for model in models_to_try:
    print(f"\n{'='*60}\n🤖 {model}\n{'='*60}")
    try:
        r = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": question}],
        )
        print(r.choices[0].message.content.strip())
    except Exception as e:
        print(f"⚠️  Failed: {e}")


🤖 llama-3.3-70b-versatile
Python is a great language for beginners due to its simple syntax, readability, and forgiving nature, making it easy to learn and understand, with a vast number of resources and libraries available to support newcomers.

🤖 llama-3.1-8b-instant
Python is a great language for beginners because it has a simple syntax, is relatively easy to read and write, and provides extensive libraries and resources that make it an accessible and forgiving language to learn.

🤖 gemma2-9b-it
⚠️  Failed: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
